# 🐛 旧代码为什么出错？—— Anthropic Tool Calling 深度解析

> 📚 **学习目标**：彻底搞懂 Anthropic API 中 `tool_use` 和 `tool_result` 的配对规则
>
> 🎯 **适合人群**：MCP / Tool Use 初学者，遇到过 `tool_use block must have a corresponding tool_result block` 报错的同学

---

## 📑 目录

1. [🧩 核心规则](#1)
2. [✅ 正确形式长什么样](#2)
3. [❌ 错误代码的问题在哪里](#3)
4. [🔍 错误流程逐步演示](#4)
5. [🛠️ 正确代码怎么解决](#5)
6. [✨ 正确流程逐步演示](#6)
7. [📝 一句话总结](#7)
8. [🧠 最重要的心智模型](#8)

<a id='1'></a>
## 🧩 1. 核心规则

Anthropic tool calling 的**硬规则**只有一句话：

> 🔑 **Assistant message 里出现了几个 `tool_use`，下一条 user message 里就必须立刻包含几个对应的 `tool_result`。**

### ✅ 合法结构

```text
assistant: tool_use A, tool_use B
user:      tool_result A, tool_result B
```

👆 一一对应，数量相等。

### ❌ 非法结构

```text
assistant: tool_use A, tool_use B
user:      tool_result A      ← 缺了 B 的 result！
assistant: ...
```

👆 这种情况下，API 会立刻报错：

```text
❗ Each tool_use block must have a corresponding tool_result block in the next message.
```

<a id='2'></a>
## ✅ 2. 正确形式长什么样

假设 Claude 一轮返回：

```text
📝 text:     "I found 2 papers. Let me extract details."
🔧 tool_use: extract_info #1
🔧 tool_use: extract_info #2
```

那么正确的 `messages` 必须长成下面这个样子 👇

In [ ]:
# ✅ 正确的 messages 结构（用于演示，不会真的运行）
messages = [
    # 1️⃣ 用户的最初提问
    {
        "role": "user",
        "content": "search physics papers and extract details"
    },

    # 2️⃣ Assistant 的完整回复（包含 text + 2 个 tool_use）
    {
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "I found 2 papers. Let me extract details."
            },
            {
                "type": "tool_use",
                "id": "toolu_A",
                "name": "extract_info",
                "input": {"paper_id": "1910.11775v2"}
            },
            {
                "type": "tool_use",
                "id": "toolu_B",
                "name": "extract_info",
                "input": {"paper_id": "hep-ex/9605011v1"}
            }
        ]
    },

    # 3️⃣ 用户回应：必须一次性把【所有】tool_result 都给齐！
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": "toolu_A",   # 👈 对应 assistant 里的 toolu_A
                "content": "paper A details..."
            },
            {
                "type": "tool_result",
                "tool_use_id": "toolu_B",   # 👈 对应 assistant 里的 toolu_B
                "content": "paper B details..."
            }
        ]
    }
]

### 🎯 三个关键点（背下来！）

| # | 规则 | 说明 |
|---|---|---|
| 1️⃣ | `tool_result.tool_use_id` 必须对应前面的 `tool_use.id` | id 错了就匹配不上 |
| 2️⃣ | 所有 `tool_result` 必须放在**下一条 user message** | 不能跨 message |
| 3️⃣ | Assistant 给了 N 个 `tool_use` → user 下一条就要给 N 个 `tool_result` | 数量必须相等 |

<a id='3'></a>
## ❌ 3. 错误代码的问题在哪里？

我们来看旧代码的关键片段 👇

In [ ]:
# ❌ 旧代码（有 bug 的写法）
for content in response.content:
    ...
    elif content.type == "tool_use":
        assistant_content.append(content)
        messages.append({"role": "assistant", "content": assistant_content})

        result = await self.session.call_tool(tool_name, arguments=tool_args)

        messages.append({
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_id,
                    "content": result.content,
                }
            ],
        })

        # 🚨 致命错误：在 for 循环内部就立刻调用 Claude！
        response = self.anthropic.messages.create(...)

### 💥 最大问题

```python
response = self.anthropic.messages.create(...)
```

这一行**写在了 `for content in response.content:` 循环内部**！

📌 这导致：**每处理一个 `tool_use`，就马上重新调用 Claude**，根本没等其他 `tool_use` 处理完。

<a id='4'></a>
## 🔍 4. 错误流程逐步演示

假设 Claude 这一轮返回了 👇

In [ ]:
# Claude 一次返回了 3 个 block：1 段文本 + 2 个 tool_use
response.content = [
    TextBlock("Great, 2 papers found..."),
    ToolUseBlock(id="toolu_A", name="extract_info", input={"paper_id": "1910.11775v2"}),
    ToolUseBlock(id="toolu_B", name="extract_info", input={"paper_id": "hep-ex/9605011v1"}),
]

### 🔄 旧代码的执行轨迹

#### Step 1️⃣：处理 text block

```python
assistant_content = [
    TextBlock("Great, 2 papers found...")
]
```

✅ 这一步没问题。

#### Step 2️⃣：处理第一个 `tool_use` (toolu_A)

把 assistant message 加进去：

In [ ]:
messages.append({
    "role": "assistant",
    "content": [
        TextBlock("Great, 2 papers found..."),
        ToolUseBlock(id="toolu_A", ...)
        # ⚠️ 注意：toolu_B 还没被加进来！
    ]
})

执行 toolu_A 工具，append user 的 tool_result：

In [ ]:
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": "toolu_A",
            "content": "paper A details..."
        }
        # ⚠️ 只有 A 的 result，没有 B 的！
    ]
})

#### Step 3️⃣：💣 立刻重新调用 Claude

```python
response = self.anthropic.messages.create(...)
```

🚨 **就在这里出事了！** 此时发给 Anthropic 的 messages 结构是👇

In [ ]:
# 🚨 发送给 API 的非法结构
messages = [
    {
        "role": "user",
        "content": "can you search for papers around physics..."
    },
    {
        "role": "assistant",
        "content": [
            {"type": "text", "text": "Great, 2 papers found..."},
            {
                "type": "tool_use",
                "id": "toolu_A",   # 🅰️
                "name": "extract_info",
                "input": {"paper_id": "1910.11775v2"}
            },
            {
                "type": "tool_use",
                "id": "toolu_B",   # 🅱️ ← 这个 tool_use 没人响应！
                "name": "extract_info",
                "input": {"paper_id": "hep-ex/9605011v1"}
            }
        ]
    },
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": "toolu_A",   # 🅰️ 只有 A 的 result
                "content": "paper A details..."
            }
            # ❌ 缺少 toolu_B 的 tool_result！
        ]
    }
]

### 💀 错误根因

```text
assistant 有 toolu_A 和 toolu_B  (2 个 tool_use)
下一条 user 只有 tool_result for toolu_A  (1 个 tool_result)
→ 缺少 toolu_B 的 tool_result
```

✋ Anthropic API 看到这个不平衡的结构，立刻报错。

<a id='5'></a>
## 🛠️ 5. 正确代码怎么解决？

正确代码做了一个**关键改变**：先把所有 tool 都执行完，再统一回应 👇

In [ ]:
# ✅ 正确写法
tool_results = []   # 🪣 准备一个收集桶

for content in response.content:
    if content.type == "text":
        print(content.text)

    elif content.type == "tool_use":
        # 🔧 执行工具
        result = await self.session.call_tool(...)
        # 🪣 把 result 放进桶里，先不发出去
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": content.id,
            "content": result_text,
        })

# 如果这一轮压根没有 tool_use，说明对话结束
if not tool_results:
    break

# 🎯 关键：循环结束后【一次性】把所有 tool_result 一起 append
messages.append({
    "role": "user",
    "content": tool_results,
})

### 🎯 重点对比

| 旧代码 ❌ | 新代码 ✅ |
|---|---|
| 执行一个 tool → append 一个 tool_result → 立刻 call Claude | 执行**所有** tools → 收集**所有** tool_results → 一次性 append → 再 call Claude |
| 在循环内部调用 Claude | 在循环外部调用 Claude |
| 一拆一发，造成不对称 | 整齐对称，一一对应 |

<a id='6'></a>
## ✨ 6. 正确流程逐步演示

Claude 同样返回 👇

In [ ]:
response.content = [
    TextBlock("Great, 2 papers found..."),
    ToolUseBlock(id="toolu_A", name="extract_info", input={"paper_id": "1910.11775v2"}),
    ToolUseBlock(id="toolu_B", name="extract_info", input={"paper_id": "hep-ex/9605011v1"}),
]

### Step 1️⃣：完整保存 assistant response（一次性！）

In [ ]:
messages.append({
    "role": "assistant",
    "content": response.content   # 👈 整体 append，不再拆
})

# 此时 messages 长这样：
# [
#     {"role": "user",      "content": "..."},
#     {"role": "assistant", "content": [Text, ToolUse_A, ToolUse_B]}   # ✅ A 和 B 一起在！
# ]

### Step 2️⃣：执行所有 tool，收集所有 result

In [ ]:
tool_results = [
    {
        "type": "tool_result",
        "tool_use_id": "toolu_A",
        "content": "paper A details..."
    },
    {
        "type": "tool_result",
        "tool_use_id": "toolu_B",
        "content": "paper B details..."
    }
]

### Step 3️⃣：一次性 append user message

In [ ]:
messages.append({
    "role": "user",
    "content": tool_results   # 👈 两个 result 整体进来
})

### 🎉 最终合法结构

In [ ]:
# ✅ Anthropic 期望的合法结构
messages = [
    {
        "role": "user",
        "content": "can you search for papers around physics and find just two of them, make sure to also extract the details"
    },
    {
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "Great, 2 papers found! Now let me extract the details."
            },
            {
                "type": "tool_use",
                "id": "toolu_A",   # 🅰️
                "name": "extract_info",
                "input": {"paper_id": "1910.11775v2"}
            },
            {
                "type": "tool_use",
                "id": "toolu_B",   # 🅱️
                "name": "extract_info",
                "input": {"paper_id": "hep-ex/9605011v1"}
            }
        ]
    },
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": "toolu_A",   # 🅰️ ✓
                "content": "paper A details..."
            },
            {
                "type": "tool_result",
                "tool_use_id": "toolu_B",   # 🅱️ ✓
                "content": "paper B details..."
            }
        ]
    }
]
# 🎯 A ↔ A, B ↔ B，完美对称！

<a id='7'></a>
## 📝 7. 一句话总结

> 旧代码的问题不是 MCP，也不是 server。
>
> ⚠️ **真正的问题是**：它把一次 assistant response 里的多个 `tool_use` **拆散处理**了。

Anthropic 的硬性要求是：

> 🔒 一个 assistant message 里的**所有** `tool_use`，必须由下一条 user message 里的**所有** `tool_result` **一次性**回应。

<a id='8'></a>
## 🧠 8. 最重要的心智模型

请把这张图刻在脑子里 🧠👇

```text
┌──────────────────────────────────────────┐
│  🎁 Claude response = 一个完整动作包       │
│                                          │
│  里面可能装着：                            │
│    📝 text                               │
│    🔧 tool_use A                         │
│    🔧 tool_use B                         │
│    🔧 tool_use C                         │
└──────────────────────────────────────────┘
                  ⬇️
      你必须【完整】处理这个动作包
                  ⬇️
┌──────────────────────────────────────────┐
│  1. 🏃 执行 A、B、C                        │
│  2. 🪣 收集 result A、B、C                 │
│  3. 📤 一次性返回所有 result              │
│  4. 🔁 然后才进入下一轮 Claude 调用       │
└──────────────────────────────────────────┘
```

### 🚫 千万不要做的事

**不要**把一次 Claude response 拆开，中途重新调用模型！

### ✅ 一定要做的事

把 response 当作**原子单位**：要么整体处理完，要么不处理。

---

## 🎓 学习检查清单

学完本笔记，你应该能回答以下问题：

- [ ] 🤔 为什么 `tool_use` 和 `tool_result` 必须数量一致？
- [ ] 🤔 旧代码的 `messages.create(...)` 写错位置导致了什么后果？
- [ ] 🤔 正确代码是怎么用一个 `tool_results` 列表解决问题的？
- [ ] 🤔 心智模型里，Claude response 应该被当作什么？

✨ 如果都能回答，恭喜你彻底搞懂了 Anthropic Tool Calling 的核心规则！🎉